In [1]:
uuid_alumno = "ff3f09cc-529b-4011-a745-256ac2565010"

# Sistema RAG Conversacional sobre Regulación e Inteligencia Artificial Agéntica

**Autor:** Alumno de Máster en Inteligencia Artificial y Ciencia de Datos  
**Asignatura:** Inteligencia Artificial Agéntica y Procesamiento del Lenguaje Natural  
**Objetivo Práctico:** Construir y auditar un motor conversacional modular de consulta legal sobre Inteligencia Artificial y Privacidad (RGPD, Guía de IA Agéntica y Dictamen del EDPB) utilizando LangGraph, ChromaDB y segmentación jerárquica sobre documentos en formato Markdown (`.md`).

---

## Preparación del Entorno y Control de Errores Inicial

En este primer bloque importamos las herramientas técnicas del ecosistema Python necesarias para construir el sistema:
* **LangChain y LangGraph:** Para cargar documentos, trocear texto estructurado y construir el grafo de conversación por nodos.
* **ChromaDB:** Para gestionar nuestra base de datos vectorial en disco local.
* **Google GenAI / Gemini:** Para vectorizar los textos (`embeddings`) y realizar la inferencia conversacional.
* **IPython.display:** Para renderizar visualmente las respuestas con formato Markdown jerárquico dentro de las celdas del cuaderno.

Como buena práctica de ingeniería de software, implementamos una comprobación de seguridad temprana para verificar que la variable de entorno `GEMINI_API_KEY` esté cargada y disponible. Si no la encuentra, el script lanza un mensaje de error claro y detiene la ejecución inmediatamente, evitando que el cuaderno falle más adelante durante la carga de vectores.


In [11]:
# Importacion de librerias requeridas para el motor RAG conversacional
import os
import time
import re
from typing import List, Dict, Any
from typing_extensions import TypedDict

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, START, END

# Herramientas de renderizado visual en Jupyter Notebook
from IPython.display import display, Markdown, HTML

# Carga automática de .env si existe en el directorio raíz
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Verificacion de la clave de acceso a la API de Gemini
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("[ERROR CRITICO]: No se ha detectado la clave GEMINI_API_KEY en las variables de entorno. Por favor, configurala antes de ejecutar el cuaderno.")
else:
    print("[INFO] Clave GEMINI_API_KEY detectada correctamente en el entorno de ejecucion.")


[INFO] Clave GEMINI_API_KEY detectada correctamente en el entorno de ejecucion.


---

## Carga del Corpus Normativo (Rutas Relativas para Portabilidad)

Cargamos los tres ficheros normativos en formato Markdown desde nuestra subcarpeta `data/raw/`:
* `EDPB_Opinion_2024_28.md` (Dictamen del Comité Europeo de Protección de Datos en inglés sobre modelos de IA).
* `Orientaciones Ia Agéntica.md` (Guía práctica en español sobre agentes autónomos y privacidad).
* `reglamentoRGPD.md` (Texto estructurado del Reglamento General de Protección de Datos).

**Regla de Portabilidad del Proyecto:** Queda estrictamente prohibido utilizar rutas absolutas locales (como `C:\Users\...` o `/home/alumno/...`). Como el cuaderno de Jupyter se ejecuta desde la carpeta `notebooks/`, utilizamos `os.path.join("..", "data", "raw")` para retroceder un directorio y llegar a la raíz de datos. De esta forma, el proyecto es totalmente portable y funcionará a la primera en el ordenador del evaluador sin necesidad de modificar rutas.

Para no sobrecargar la vista del cuaderno con cientos de líneas de texto plano, imprimimos únicamente una tabla resumen que verifica la correcta lectura de cada archivo y muestra su peso en KB y cantidad de caracteres.


In [3]:
# Carga limpia y estructurada utilizando rutas relativas
# Al estar el cuaderno situado en la subcarpeta /notebooks, retrocedemos un nivel con '..'
ruta_raw = os.path.join("..", "data", "raw")

archivos_md = [
    "EDPB_Opinion_2024_28.md",
    "Orientaciones Ia Agéntica.md",
    "reglamentoRGPD.md"
]

documentos_crudos = []
print("[INFO] Cargando ficheros normativos desde la carpeta data/raw/:")
print(f"{'Archivo normativo':<35} | {'Tamano (KB)':<12} | {'Caracteres':<12}")
print("-" * 65)

for archivo in archivos_md:
    ruta_completa = os.path.join(ruta_raw, archivo)
    if not os.path.exists(ruta_completa):
        raise FileNotFoundError(f"[ERROR]: No se encuentra el archivo {archivo} en {ruta_raw}. Verifica la estructura de carpetas.")
    
    loader = TextLoader(ruta_completa, encoding="utf-8")
    docs = loader.load()
    documentos_crudos.extend(docs)
    
    tamano_kb = os.path.getsize(ruta_completa) / 1024
    num_caracteres = len(docs[0].page_content)
    print(f"{archivo:<35} | {tamano_kb:<12.2f} | {num_caracteres:<12}")

print("-" * 65)
print(f"[INFO] Carga completada con exito. Documentos procesados: {len(documentos_crudos)}")


[INFO] Cargando ficheros normativos desde la carpeta data/raw/:
Archivo normativo                   | Tamano (KB)  | Caracteres  
-----------------------------------------------------------------
EDPB_Opinion_2024_28.md             | 93.83        | 96079       
Orientaciones Ia Agéntica.md        | 41.74        | 42085       
reglamentoRGPD.md                   | 388.82       | 388886      
-----------------------------------------------------------------
[INFO] Carga completada con exito. Documentos procesados: 3


---

## Troceado Semántico y Jerárquico en Dos Etapas (Two-Stage Chunking)

Para transformar las leyes completas en fragmentos útiles para el buscador vectorial sin perder la estructura de la norma, aplicamos un procedimiento de segmentación en dos etapas:

* **Etapa 1 (`MarkdownHeaderTextSplitter`):** Le indicamos al cortador que reconozca tres niveles jerárquicos de cabeceras Markdown:
  * `#` para extraer y etiquetar el `Titulo_Ley`
  * `##` para extraer y etiquetar el `Capitulo` o sección principal
  * `###` para extraer y etiquetar el `Articulo` o subsección
  Al procesar los textos, este divisor corta el documento por las fronteras exactas de las secciones y guarda esas etiquetas dentro de los metadatos de cada bloque. Mantenemos el parámetro `strip_headers=False` para que el título también permanezca escrito dentro del propio texto del trozo, reforzando la señal temática para el vector de embedding.

* **Etapa 2 (`RecursiveCharacterTextSplitter`):** En textos normativos reales existen artículos muy breves y otros extensísimos (como los artículos de multas o sanciones del RGPD). Si mandamos un bloque gigante a vectorizar, el modelo pierde precisión semántica al promediar ideas diversas. Por ello, pasamos los bloques resultantes de la Etapa 1 por un segundo divisor configurado a 1.000 caracteres como máximo, con un margen de solapamiento (`overlap`) de 150 caracteres para que las frases y párrafos mantengan continuidad estructural. La ventaja clave de este diseño es que la segunda etapa conserva intacta toda la genealogía jerárquica de metadatos obtenida en la primera.

Al ejecutar la celda mostramos el conteo total de fragmentos resultantes y una muestra técnica de los metadatos generados para verificar su correcta estructuración.


In [4]:
# Etapa 1: Corte estructural respetando los encabezados legales
headers_to_split_on = [
    ("#", "Titulo_Ley"),
    ("##", "Capitulo"),
    ("###", "Articulo"),
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False  # Conservamos la cabecera dentro del texto para enriquecer la busqueda semantica
)

chunks_estructurales = []
for doc in documentos_crudos:
    splits = md_splitter.split_text(doc.page_content)
    for s in splits:
        # Aseguramos un nombre de archivo limpio en los metadatos
        s.metadata["source"] = os.path.basename(doc.metadata.get("source", "desconocido"))
    chunks_estructurales.extend(splits)

# Etapa 2: Corte por tamano maximo con doble barra invertida en saltos de linea para evitar errores de sintaxis al exportar
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_finales = char_splitter.split_documents(chunks_estructurales)

print("[INFO] Resumen de la segmentacion semantica:")
print(f"       - Bloques tras corte estructural por cabeceras (Etapa 1): {len(chunks_estructurales)}")
print(f"       - Trozos finales listos para indexar (Etapa 2): {len(chunks_finales)}")
print("-" * 65)
print("[INFO] Ejemplo de metadatos jerarquicos generados en un trozo de muestra:")
muestra_idx = min(50, len(chunks_finales) - 1)
for clave, valor in chunks_finales[muestra_idx].metadata.items():
    print(f"       * {clave}: {valor}")


[INFO] Resumen de la segmentacion semantica:
       - Bloques tras corte estructural por cabeceras (Etapa 1): 293
       - Trozos finales listos para indexar (Etapa 2): 903
-----------------------------------------------------------------
[INFO] Ejemplo de metadatos jerarquicos generados en un trozo de muestra:
       * Titulo_Ley: **Opinion 28/2024 on certain data protection aspects related to the processing of personal data in the context of AI models**
       * Capitulo: **3 On the merits of the request**
       * Articulo: **3.1 On the nature of AI models in relation to the definition of personal data**
       * source: EDPB_Opinion_2024_28.md


---

## Base de Datos Vectorial Local y Control de Cuotas de API

En este bloque creamos la base de datos vectorial donde residirá todo el conocimiento normativo procesado. Para ello utilizamos **ChromaDB** guardado localmente en disco, y el modelo de vectorización `models/gemini-embedding-001` de Google AI Studio.

Durante las pruebas prácticas de indexación nos encontramos con un desafío técnico habitual al operar en cuentas gratuitas en la nube: si enviamos los casi 300 trozos de texto a vectorizar todos de golpe, la API rechaza las peticiones por superar el límite máximo de peticiones por minuto (error 429 por saturación o cuota de rate-limiting). Para solucionarlo y garantizar un script estable y tolerante a fallos, implementamos tres defensas técnicas:

* **Ruta relativa de persistencia en disco (`../vector_db`):** Toda la base vectorial se persiste en la carpeta local `vector_db` situada en el directorio raíz del proyecto.
* **Comprobación condicional inteligente:** Antes de iniciar llamadas de red, el script consulta si la colección `corpus_normativo_v3` ya existe en disco y tiene vectores almacenados. Si ya existe y la variable de control `forzar_reindexacion` es `False`, se omite por completo el bucle de embedding. Así cargamos la colección en milisegundos sin consumir peticiones de la cuota diaria.
* **Bucle de indexación por lotes reducidos con resiliencia de tolerancia a fallos (`Backoff reactivo`):** Cuando se indexa desde cero o cuando se fuerza la reindexación (`forzar_reindexacion = True`), el sistema agrupa los fragmentos en lotes de 15 trozos intercalando una pausa ligera de 2 segundos entre lotes. Al operar sobre la capa gratuita de Google AI Studio (`Free Tier`, con límite estricto de 100 peticiones por minuto), es esperable que durante la vectorización masiva de los 293 fragmentos se alcancen picos temporales de saturación (`Error 429 RESOURCE_EXHAUSTED`). Nuestro bloque defensivo intercepta estos eventos, informa por pantalla e inyecta exactamente 65 segundos de pausa para reiniciar el reloj de cuota móvil de Google antes de reintentar el lote (mostrando el progreso `Reintento 1/3 para este lote concreto`). Gracias a esta arquitectura de resiliencia y tolerancia a fallos, el sistema se recupera de forma autónoma y finaliza la indexación al 100% (293 vectores persistidos) sin que el kernel de Python colapse ni se pierda un solo dato.


In [5]:
# Configuracion del modelo de embeddings de Gemini
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

# Definicion de la ruta relativa local en ../vector_db
ruta_chroma = os.path.join("..", "vector_db")
vectorstore = Chroma(
    collection_name="corpus_normativo_v3",
    embedding_function=embeddings,
    persist_directory=ruta_chroma
)

# Control condicional de persistencia para proteger la cuota de la API
forzar_reindexacion = False  # Cambia a True si editas algun archivo en data/raw/ y necesitas regenerar los vectores
vectores_existentes = vectorstore._collection.count()

if vectores_existentes > 0 and not forzar_reindexacion:
    print(f"[EXITO] Base de datos vectorial detectada en disco ({ruta_chroma}).")
    print(f"[INFO] Se han cargado directamente {vectores_existentes} vectores pre-indexados en memoria.")
    print("[INFO] Se omite el bucle de embedding para ahorrar tiempo y cuota diaria de peticiones de la API.")
else:
    if forzar_reindexacion and vectores_existentes > 0:
        print("[INFO] forzar_reindexacion=True: Limpiando coleccion anterior en ChromaDB...")
        vectorstore.delete_collection()
        vectorstore = Chroma(
            collection_name="corpus_normativo_v3",
            embedding_function=embeddings,
            persist_directory=ruta_chroma
        )
    
    print("[INFO] Iniciando indexacion vectorial en ChromaDB mediante lotes controlados...")
    t_inicio = time.time()
    
    batch_size = 15  # Lotes reducidos de 15 trozos para no exceder los tokens por minuto
    for i in range(0, len(chunks_finales), batch_size):
        lote = chunks_finales[i : i + batch_size]
        # El contador de reintentos se reinicia intencionalmente a 0 para cada lote
        # independiente, permitiendo hasta 3 intentos por cada bloque de 15 trozos.
        reintentos = 0
        max_reintentos = 3
        
        while reintentos <= max_reintentos:
            try:
                vectorstore.add_documents(lote)
                break
            except Exception as e:
                reintentos += 1
                if reintentos > max_reintentos:
                    print(f"[ERROR CRITICO] Fallo la indexacion del lote {i} tras {max_reintentos} reintentos: {e}")
                    raise e
                
                # Pausa prudencial de 65s ante saturacion 429 para permitir el reinicio de la cuota del servidor
                tiempo_espera = 65.0
                print(f"[ADVERTENCIA] Error de cuota o saturacion de red ({e}). Esperando {tiempo_espera}s para reiniciar el contador de Google (Reintento {reintentos}/{max_reintentos} para este lote concreto)...")
                time.sleep(tiempo_espera)
        
        # Pausa ligera entre lotes en condiciones normales
        time.sleep(2.0)
    
    t_total = time.time() - t_inicio
    print(f"[EXITO] Indexacion completada. {len(chunks_finales)} vectores persistidos en {t_total:.2f} segundos.")


[EXITO] Base de datos vectorial detectada en disco (..\vector_db).
[INFO] Se han cargado directamente 903 vectores pre-indexados en memoria.
[INFO] Se omite el bucle de embedding para ahorrar tiempo y cuota diaria de peticiones de la API.


---

## Verificación Aislada del Motor de Búsqueda Vectorial (Paso 2 del Enunciado)

Antes de acoplar el índice vectorial al grafo de LangGraph o conectar el modelo de lenguaje (LLM), el Paso 2 de las especificaciones exige **verificar de manera independiente que la colección vectorial en ChromaDB responde correctamente a consultas de prueba en crudo (`similarity_search`)**.

Esta prueba confirma dos aspectos críticos del sistema de recuperación:
1. **Integridad del Índice:** Que el buscador recupera los fragmentos normativos adecuados utilizando la distancia del coseno sin depender de la inferencia del LLM.
2. **Preservación Jerárquica:** Que cada documento recuperado conserva intactos los metadatos de su árbol normativo (`source`, `Titulo_Ley`, `Capitulo`, `Articulo`) para alimentar con precisión la trazabilidad jurídica posterior.


In [6]:
# Verificacion aislada del retriever en crudo (Sin intervencion del LLM)
consulta_test = "¿En qué condiciones es lícito el tratamiento de datos personales según el RGPD y qué papel juega el consentimiento?"
print(f"[PASO 2 - AUDITORIA DE RECUPERACION] Ejecutando busqueda vectorial de prueba en crudo (top_k=2)...")
print(f"Consulta de control: '{consulta_test}'\n" + "="*75)

resultados_test = vectorstore.similarity_search(consulta_test, k=2)

if resultados_test:
    for idx, doc in enumerate(resultados_test, 1):
        fuente = doc.metadata.get("source", "desconocido")
        titulo = doc.metadata.get("Titulo_Ley", "")
        capitulo = doc.metadata.get("Capitulo", "")
        articulo = doc.metadata.get("Articulo", "")
        jerarquia = f"{titulo} -> {capitulo} -> {articulo}".strip(" ->")
        
        print(f"\n[RESULTADO #{idx}] Fuente: {fuente} | Estructura: [{jerarquia}]")
        print(f"Extracto del fragmento recuperado (primeros 250 caracteres):")
        print("-" * 75)
        print(doc.page_content[:250].replace("\n", " ") + "...")
        print("-" * 75)
    print("\n[EXITO VERIFICADO] El indice vectorial responde correctamente y conserva los metadatos jerarquicos.")
else:
    print("[ERROR] No se recuperaron documentos en la prueba aislada del retriever.")


[PASO 2 - AUDITORIA DE RECUPERACION] Ejecutando busqueda vectorial de prueba en crudo (top_k=2)...
Consulta de control: '¿En qué condiciones es lícito el tratamiento de datos personales según el RGPD y qué papel juega el consentimiento?'

[RESULTADO #1] Fuente: reglamentoRGPD.md | Estructura: [REGLAMENTOS -> **de 27 de abril de 2016**]
Extracto del fragmento recuperado (primeros 250 caracteres):
---------------------------------------------------------------------------
- (40) Para que el tratamiento sea lícito, los datos personales deben ser tratados con el consentimiento del interesado o sobre alguna otra base legítima establecida conforme a Derecho, ya sea en el presente Reglamento o en virtud de   <u>ES</u>   ot...
---------------------------------------------------------------------------

[RESULTADO #2] Fuente: reglamentoRGPD.md | Estructura: [REGLAMENTOS -> **de 27 de abril de 2016**]
Extracto del fragmento recuperado (primeros 250 caracteres):
----------------------------------

---

## Motor RAG Conversacional Orquestado con LangGraph (3 Nodos)

Para gestionar la memoria de las conversaciones, resolver consultas multi-turno y orquestar las búsquedas en el índice vectorial, construimos un grafo de estado (`StateGraph`) con **LangGraph**.

### Justificación Técnica de la Arquitectura de 3 Nodos (`rewrite_query` -> `retrieve` -> `generate`):
Al diseñar la estructura del grafo evaluamos la inclusión de nodos de reflexión iterativa o autoevaluación reflexiva del contexto por parte del LLM. Sin embargo, tras analizar el comportamiento en un corpus normativo acotado y estructurado en Markdown, concluimos que añadir bucles reflexivos duplicaba innecesariamente el consumo de tokens y aumentaba notablemente la latencia de inferencia sin aportar mejoras medibles en la fidelidad jurídica. Por este motivo, priorizando la estabilidad técnico-legal, el determinismo normativo y la protección de la cuota gratuita de la API, elegimos una arquitectura lineal y altamente eficiente compuesta por **3 nodos especializados en secuencia**:
* **Nodo `rewrite_query` (Reescritura conversacional):** Analiza el historial de turnos recientes para detectar si el usuario utiliza pronombres, elipsis o referencias implícitas (por ejemplo: *"¿Y qué multas se aplican en ese caso?"*). Si detecta dependencias conversacionales, reescribe la pregunta para que sea completamente autónoma y el buscador vectorial localice los artículos correctos.
* **Nodo `retrieve` (Búsqueda defensiva en ChromaDB):** Toma la consulta limpia y extrae los 4 fragmentos normativos con mayor similitud del coseno (`top_k=4`) desde `vector_db`, gestionando posibles fallos de conexión sin interrumpir el programa.
* **Nodo `generate` (Razonamiento legal y Guardrails):** Recibe el contexto recuperado, el historial y la consulta del usuario, y genera la respuesta fundamentada respetando estrictamente los controles de seguridad y formato del prompt.

### Selección del Modelo LLM (`gemini-3.1-flash-lite` y Temperatura 0.2):
* Al procesar cada pregunta del chat a través del nodo reformulador y del nodo generador, ejecutamos al menos dos llamadas al modelo de inferencia por turno. Elegir **`gemini-3.1-flash-lite`** resulta óptimo porque nos brinda una velocidad de inferencia casi instantánea y un margen muy amplio en el límite de peticiones por minuto (RPM) en la capa gratuita de Google AI Studio.
* Establecemos la **temperatura en 0.2**: este ligero aumento sobre el cero puro permite que el modelo redacte dictámenes y resúmenes normativos con naturalidad gramatical y riqueza expresiva, manteniendo al mismo tiempo un determinismo estricto para no desviar la información legal ni generar alucinaciones.

### Buenas Prácticas Senior: Gestión de Memoria y Ventana Deslizante (`Sliding Window`):
En este proyecto, el nodo reformulador (`rewrite_query_node`) ya aplica una ventana deslizante inteligente cogiendo exclusivamente los últimos 3 turnos (`messages[-4:-1]`) para resolver pronombres y elipsis. En el nodo generador (`generate_node`), durante la evaluación estándar el historial se inyecta completo. Para un paso a producción masiva o sesiones de chat prolongadas (15+ turnos), se recomienda formalmente acotar el historial pasado a la generación mediante una **Ventana Deslizante de Memoria (`Sliding Window` de los últimos 6 turnos)** para mantener la precisión de atención del LLM y optimizar el gasto de tokens sin diluir el contexto legal.

### Blindaje Estratégico y Guardrails Integrados en el System Prompt:
Para asegurar el rigor auditor de la herramienta y prevenir respuestas incorrectas o ambiguas, configuramos el prompt maestro `template_rag` con cinco protecciones y reglas formales:
* **Guardrail de Ausencia de Información:** Si el buscador vectorial devuelve un conjunto vacío (porque la materia no figura en la ley), el nodo generador intercepta el flujo y responde con honestidad que la base de conocimientos no permite responder a esa consulta.
* **Guardrail de Dominio:** Si el usuario pregunta sobre temas ajenos a la ley o la tecnología (como recetas de cocina o deportes), el modelo rechaza la pregunta indicando que su especialización abarca exclusivamente el ámbito legal y normativo.
* **Guardrail de Idioma Multilingüe:** Obliga al consultor a detectar el idioma exacto en el que el usuario formula su consulta (por ejemplo, en inglés al preguntar por el dictamen del EDPB) y a responder al 100% en ese mismo idioma, traduciendo de forma fidedigna los artículos recuperados si estuviesen originalmente en español.
* **Regla de Desduplicación de Citas:** Exige que si varios de los fragmentos leídos provienen de un mismo artículo o capítulo de una ley, se agrupen en una sola entrada en la lista final para evitar repeticiones visuales innecesarias.
* **Formato Estructurado sin Numeraciones:** Indicamos al modelo que presente su salida en tres bloques limpios con encabezados Markdown sin numerar (`### Conclusión Directa`, `### Análisis Normativo` y `### Trazabilidad Jurídica`).


In [7]:
# Definicion del estado para el grafo de LangGraph
class GraphState(TypedDict):
    messages: List[Any]
    query_reformulada: str
    documentos_recuperados: List[Any]

# Instanciacion del modelo LLM conversacional agil
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.2,
    max_tokens=1500,
    max_retries=6,
    google_api_key=api_key
)

# NODO 1: Reformulacion de preguntas para resolver anaforas o elipsis del historial
def rewrite_query_node(state: GraphState) -> Dict[str, Any]:
    messages = state["messages"]
    ultima_pregunta = messages[-1].content if messages else ""
    
    if len(messages) <= 1:
        return {"query_reformulada": ultima_pregunta}
    
    prompt_reformulacion = f"""Historial reciente del chat:
{[m.content for m in messages[-4:-1]]}

Pregunta actual del usuario: "{ultima_pregunta}"

Tu tarea tecnica es reformular la pregunta actual para que sea independiente y autocontenida, reemplazando pronombres o referencias implicitas por los conceptos de los turnos anteriores. Devuelve unicamente la pregunta reformulada sin preambulos ni explicaciones."""

    respuesta = llm.invoke([HumanMessage(content=prompt_reformulacion)])
    contenido_req = respuesta.content
    if isinstance(contenido_req, list):
        contenido_req = "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in contenido_req])
    return {"query_reformulada": contenido_req.strip()}

# NODO 2: Recuperacion de documentos relevantes en ChromaDB
def retrieve_node(state: GraphState) -> Dict[str, Any]:
    query = state.get("query_reformulada") or state["messages"][-1].content
    try:
        retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
        docs = retriever.invoke(query)
        return {"documentos_recuperados": docs}
    # Interceptacion generica de excepciones para blindar la ejecucion academica del grafo;
    # en un paso a produccion se segmentarian errores de API de red frente a fallos de I/O en disco local.
    except Exception as e:
        print(f"[ADVERTENCIA] Error en la consulta de busqueda vectorial: {e}")
        return {"documentos_recuperados": []}

# NODO 3: Generacion fundamentada con Guardrails, Idioma y Trazabilidad Limpia
template_rag = """[PERFIL Y TONO]
Eres un Consultor Legal Senior y Arquitecto de Cumplimiento Normativo en Inteligencia Artificial y Privacidad (RGPD / EDPB / IA Agéntica).
Tu lenguaje debe ser formal, preciso, analítico, riguroso y estrictamente acotado al conocimiento normativo facilitado.

[REGLAS DE COMPORTAMIENTO ESTRATEGICO Y GUARDRAILS]
1. Responde UNICAMENTE utilizando la información técnica y normativa presente en los fragmentos de [CONTEXTO DE OPERACION RECUPERADO]. No inventes ni supongas nada fuera de estos textos.
2. Si el usuario hace referencia a una pregunta o respuesta anterior, consulta el [HISTORIAL DE CONVERSACION RECIENTE] para mantener coherencia absoluta.
3. GUARDRAIL DE DOMINIO: Si la pregunta del usuario NO está relacionada en absoluto con el ámbito legal, tecnológico, inteligencia artificial o protección de datos (ej. recetas de cocina, deportes, literatura general), debes responder exactamente: "No estoy entrenado para responder sobre ese tema".
4. GUARDRAIL DE AUSENCIA DE INFORMACION: Si la pregunta está dentro del dominio jurídico/tecnológico, pero la respuesta no se encuentra en los fragmentos recuperados, tu única respuesta debe ser: "La información disponible en la base de conocimientos no permite responder a esta consulta".
5. GUARDRAIL DE IDIOMA (ESPEJO LINGÜÍSTICO OBLIGATORIO): Detecta el idioma en que está escrita la "Pregunta actual del usuario" y redacta tu dictamen SIEMPRE en ese exacto mismo idioma:
   - Si la "Pregunta actual del usuario" está en ESPAÑOL, el 100% de tu respuesta (Conclusión Directa, Análisis Normativo y Trazabilidad Jurídica) DEBE estar estrictamente en ESPAÑOL. Queda terminantemente prohibido usar inglés.
   - Si la "Pregunta actual del usuario" está en INGLÉS, el 100% de tu dictamen DEBE redactarse estrictamente en INGLÉS, traduciendo al inglés cualquier concepto legal de los textos en español.

[FORMATO DE SALIDA OBLIGATORIO]
Estructura tu respuesta estrictamente utilizando encabezados Markdown reales SIN NUMERAR y los siguientes tres bloques diferenciados:

### Conclusión Directa
Una única frase corta, profesional y categórica respondiendo de forma clara e inequívoca a la consulta planteada.

### Análisis Normativo
Explicación técnica detallada y fundamentada en viñetas ordenadas o párrafos breves, basada exclusivamente en los preceptos y considerandos de los fragmentos recuperados.

### Trazabilidad Jurídica
Indica al final de tu respuesta las fuentes exactas de donde extrajiste cada afirmación. REGLA DE DESDUPLICACION OBLIGATORIA: Si varios fragmentos utilizados pertenecen exactamente a la misma ley, capítulo y artículo o sección, agrupa las referencias y escribe esa cabecera UNA SOLA VEZ en la lista final. Está estrictamente prohibido repetir citas legales idénticas. Utiliza viñetas tipográficas limpias con el formato:
* **<Nombre_Archivo> (<Titulo_Ley> -> <Capitulo> -> <Articulo>)** — Breve mención al precepto.

[HISTORIAL DE CONVERSACION RECIENTE]
{chat_history}

[CONTEXTO DE OPERACION RECUPERADO]
{context}

Pregunta actual del usuario: 
{question}
"""

def generate_node(state: GraphState) -> Dict[str, Any]:
    messages = state["messages"]
    docs = state.get("documentos_recuperados", [])
    query = state.get("query_reformulada") or messages[-1].content
    
    if not docs:
        msg_fallo = "La información disponible en la base de conocimientos no permite responder a esta consulta."
        return {"messages": messages + [AIMessage(content=msg_fallo)]}
    
    contexto_estratificado = []
    for d in docs:
        src = d.metadata.get("source", "desconocido")
        tit = d.metadata.get("Titulo_Ley", "")
        cap = d.metadata.get("Capitulo", "")
        art = d.metadata.get("Articulo", "")
        jerarquia = f"{tit} -> {cap} -> {art}".strip(" ->")
        if not jerarquia:
            jerarquia = "Seccion Normativa General"
        
        contexto_estratificado.append(f"--- DOCUMENTO: {src} | ESTRUCTURA: [{jerarquia}] ---\n{d.page_content}")
        
    texto_contexto = "\n\n".join(contexto_estratificado)
    
    historial_str = ""
    if len(messages) > 1:
        # Ventana deslizante de memoria (Sliding Window): retenemos únicamente los últimos 4 turnos (2 intercambios)
        # para evitar saturación del contexto e impedir que idiomas o temas pasados interfieran con el turno actual.
        turnos_recientes = messages[-5:-1] if len(messages) > 5 else messages[:-1]
        for m in turnos_recientes:
            rol = "Usuario" if isinstance(m, HumanMessage) else "Consultor"
            historial_str += f"{rol}: {m.content}\n"
            
    prompt_evaluado = template_rag.format(
        chat_history=historial_str or "Sin historial previo.",
        context=texto_contexto,
        question=query
    )
    
    respuesta = llm.invoke([HumanMessage(content=prompt_evaluado)])
    contenido = respuesta.content
    if isinstance(contenido, list):
        contenido = "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in contenido])
    return {"messages": messages + [AIMessage(content=contenido)]}

# Ensamblaje del grafo con LangGraph
workflow = StateGraph(GraphState)
workflow.add_node("rewrite_query", rewrite_query_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)

workflow.add_edge(START, "rewrite_query")
workflow.add_edge("rewrite_query", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

app_rag = workflow.compile()
print("[INFO] Motor RAG conversacional con LangGraph compilado y listo en memoria.")


[INFO] Motor RAG conversacional con LangGraph compilado y listo en memoria.


---

## Suite de Evaluación Automatizada y Pruebas de Esfuerzo (Stress Testing)

Para auditar de forma sistemática y reproducible el funcionamiento de la arquitectura, ejecutamos una batería automatizada de pruebas de esfuerzo compuesta por **6 casos de control estratégicos**:

* **Consulta directa sobre RGPD:** Comprueba la exactitud en la recuperación de condiciones de licitud y consentimiento.
* **Razonamiento multi-turno con anáfora:** Verifica que el nodo reformulador reescribe correctamente una consulta dependiente (*"¿Y qué multas administrativas se pueden imponer...?"*) enlazándola con el contexto del turno previo.
* **Consulta técnico-legal sobre IA Agéntica:** Evalúa la capacidad del sistema para analizar riesgos de privacidad asociados a patrones técnicos como la cadena de razonamiento (*Chain of Thought*).
* **Consulta multilingüe en inglés (Dictamen EDPB):** Confirma que el Guardrail #5 detecta el idioma original de la consulta en inglés y redacta su dictamen 100% en ese idioma, traduciendo de forma exacta los conceptos del corpus.
* **Guardrail de Dominio (Prueba fuera de ámbito):** Intentamos desviar la atención del modelo consultando por una receta de paella valenciana tradicional. El agente debe interceptar la consulta y responder el mensaje exacto de rechazo.
* **Guardrail de Ausencia de Información:** Preguntamos por multas para drones agrícolas en Australia (materia legal pero no presente en nuestro corpus). El sistema debe reconocer sus límites de información sin alucinar o inventar respuestas.

Para lograr un acabado visual limpio en Jupyter, renderizamos las salidas utilizando `display(Markdown(...))`, aprovechando las listas tipográficas y los encabezados sin numerar.


In [8]:
# Suite de 6 pruebas practicas para validar el grafo y sus guardrails en condiciones reales
consultas_evaluacion = [
    {
        "pregunta": "¿En qué condiciones es lícito el tratamiento de datos personales según el RGPD y qué papel juega el consentimiento?",
        "tipo": "Normativa RGPD (Consulta Directa)",
        "reiniciar_historial": True
    },
    {
        "pregunta": "¿Y qué multas administrativas se pueden imponer si no se cumplen esas condiciones que acabas de mencionar?",
        "tipo": "Razonamiento Multi-turno (Anáfora y Memoria)",
        "reiniciar_historial": False
    },
    {
        "pregunta": "¿Qué riesgos para la privacidad presenta el patrón de diseño de 'cadena de razonamiento' en los agentes de IA?",
        "tipo": "IA Agéntica (Patrones de Diseño y Riesgos)",
        "reiniciar_historial": True
    },
    {
        "pregunta": "According to the EDPB Opinion 2024/28, what are the key considerations regarding the lawful basis for processing personal data to train AI models?",
        "tipo": "Dictamen EDPB (Multilingüe en Inglés)",
        "reiniciar_historial": True
    },
    {
        "pregunta": "¿Cuál es la receta para preparar una paella valenciana tradicional y qué ingredientes lleva?",
        "tipo": "Guardrail de Seguridad (Fuera de Dominio)",
        "reiniciar_historial": True
    },
    {
        "pregunta": "¿Cuál es la regulación específica y las multas para el uso de drones agrícolas en la legislación de Australia?",
        "tipo": "Guardrail de Seguridad (Ausencia de Información)",
        "reiniciar_historial": True
    }
]

print("=" * 68)
print("   AUDITORIA Y PRUEBAS DE ESFUERZO DEL MOTOR RAG (MARKDOWN)")
print("=" * 68 + "\n")

historial_actual = []

for idx, test in enumerate(consultas_evaluacion, 1):
    pregunta = test["pregunta"]
    tipo = test["tipo"]
    reiniciar = test["reiniciar_historial"]
    
    if reiniciar:
        historial_actual = []
        
    print(f"--- PRUEBA {idx}/6: [{tipo}] ---")
    print(f"Pregunta Usuario: {pregunta}")
    
    historial_actual.append(HumanMessage(content=pregunta))
    
    t0 = time.time()
    estado_input = {"messages": historial_actual}
    estado_salida = app_rag.invoke(estado_input)
    t_duracion = time.time() - t0
    
    historial_actual = estado_salida["messages"]
    respuesta_generada = historial_actual[-1].content
    
    print(f"Tiempo de inferencia: {t_duracion:.2f}s")
    print("-" * 68)
    
    texto_md_eval = respuesta_generada if isinstance(respuesta_generada, str) else "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in respuesta_generada])
    display(Markdown(texto_md_eval))
    print("\n" + "=" * 68 + "\n")

print("[EXITO] Suite de evaluacion completada. Todas las pruebas del sistema han finalizado correctamente.")


   AUDITORIA Y PRUEBAS DE ESFUERZO DEL MOTOR RAG (MARKDOWN)

--- PRUEBA 1/6: [Normativa RGPD (Consulta Directa)] ---
Pregunta Usuario: ¿En qué condiciones es lícito el tratamiento de datos personales según el RGPD y qué papel juega el consentimiento?
Tiempo de inferencia: 2.54s
--------------------------------------------------------------------


### Conclusión Directa
El tratamiento de datos personales es lícito únicamente cuando se fundamenta en una base jurídica válida, siendo el consentimiento una de ellas, siempre que cumpla con requisitos estrictos de libertad, especificidad e informabilidad.

### Análisis Normativo
* **Bases de licitud:** El tratamiento de datos personales solo es conforme a derecho si se ajusta a al menos una de las siguientes condiciones:
    * El interesado ha otorgado su consentimiento para fines específicos.
    * Es necesario para la ejecución de un contrato o medidas precontractuales.
    * Es necesario para el cumplimiento de una obligación legal del responsable.
    * Es necesario para proteger intereses vitales del interesado o de terceros.
    * Es necesario para el cumplimiento de una misión de interés público o ejercicio de poderes públicos.
* **Naturaleza del consentimiento:** Para que el consentimiento sea un fundamento jurídico válido, debe ser otorgado libremente. Se presume que no es libre cuando existe un desequilibrio claro entre las partes (especialmente con autoridades públicas) o cuando la ejecución de un contrato se condiciona al consentimiento de operaciones no necesarias para dicho cumplimiento.
* **Condiciones operativas:** El responsable debe ser capaz de demostrar que el interesado consintió. Si el consentimiento se incluye en una declaración escrita, debe distinguirse claramente de otros asuntos mediante un lenguaje sencillo. Asimismo, el interesado tiene el derecho de retirar su consentimiento en cualquier momento, debiendo ser este proceso tan sencillo como el de su otorgamiento.

### Trazabilidad Jurídica
* **reglamentoRGPD.md (REGLAMENTOS -> Licitud del tratamiento)** — Establece las condiciones taxativas para la licitud del tratamiento.
* **reglamentoRGPD.md (REGLAMENTOS -> de 27 de abril de 2016)** — Define los fundamentos de licitud (considerando 40) y las restricciones sobre la libertad del consentimiento y la necesidad contractual (considerandos 43 y 44).
* **reglamentoRGPD.md (REGLAMENTOS -> Condiciones para el consentimiento)** — Detalla las obligaciones del responsable respecto a la demostración, forma y revocabilidad del consentimiento.



--- PRUEBA 2/6: [Razonamiento Multi-turno (Anáfora y Memoria)] ---
Pregunta Usuario: ¿Y qué multas administrativas se pueden imponer si no se cumplen esas condiciones que acabas de mencionar?
Tiempo de inferencia: 2.13s
--------------------------------------------------------------------


### Conclusión Directa
La información disponible en la base de conocimientos no permite responder a esta consulta.

### Análisis Normativo
Los fragmentos recuperados detallan los criterios, límites máximos y factores agravantes o atenuantes para la imposición de multas administrativas en términos generales, así como las sanciones específicas por incumplimiento de obligaciones de responsables, encargados, organismos de certificación y resoluciones de la autoridad de control. Sin embargo, no existe en la documentación proporcionada una vinculación explícita que tipifique las sanciones específicas derivadas del incumplimiento de las condiciones de licitud del tratamiento o de los requisitos del consentimiento.

### Trazabilidad Jurídica
* **reglamentoRGPD.md (REGLAMENTOS -> Condiciones generales para la imposición de multas administrativas)** — Establece los umbrales económicos de las sanciones y los factores de ponderación.
* **reglamentoRGPD.md (REGLAMENTOS -> de 27 de abril de 2016)** — Define los criterios de individualización de la multa por parte de la autoridad de control.



--- PRUEBA 3/6: [IA Agéntica (Patrones de Diseño y Riesgos)] ---
Pregunta Usuario: ¿Qué riesgos para la privacidad presenta el patrón de diseño de 'cadena de razonamiento' en los agentes de IA?
Tiempo de inferencia: 2.97s
--------------------------------------------------------------------


...



--- PRUEBA 5/6: [Guardrail de Seguridad (Fuera de Dominio)] ---
Pregunta Usuario: ¿Cuál es la receta para preparar una paella valenciana tradicional y qué ingredientes lleva?
Tiempo de inferencia: 0.77s
--------------------------------------------------------------------


No estoy entrenado para responder sobre ese tema.



--- PRUEBA 6/6: [Guardrail de Seguridad (Ausencia de Información)] ---
Pregunta Usuario: ¿Cuál es la regulación específica y las multas para el uso de drones agrícolas en la legislación de Australia?
Tiempo de inferencia: 0.79s
--------------------------------------------------------------------


No estoy entrenado para responder sobre ese tema.



[EXITO] Suite de evaluacion completada. Todas las pruebas del sistema han finalizado correctamente.


---

## Celda Interactiva de Conversación en Vivo (Paso 6 del Enunciado)

Para dar cumplimiento al **Paso 6 del Enunciado Práctico**, incorporamos a continuación una celda ejecutable con un bucle conversacional interactivo. Esta celda permite al profesor, evaluador o estudiante realizar consultas libres en tiempo real directamente desde la interfaz del Jupyter Notebook.

### Funcionamiento del Bucle Interactivo:
* Al ejecutar la celda, el sistema inicializa un historial en blanco y solicita una pregunta por teclado mediante la función `input()`.
* La consulta atraviesa los tres nodos del grafo: reescritura de posibles referencias previas, búsqueda en la base vectorial de `ChromaDB` y redacción fundamentada por el modelo `Gemini Flash-Lite`.
* La respuesta del Consultor Legal se visualiza al momento renderizada en formato Markdown (`display(Markdown(...))`), organizada en sus tres encabezados jerárquicos sin numerar (`### Conclusión Directa`, `### Análisis Normativo` y `### Trazabilidad Jurídica`) y con sus fuentes desduplicadas.
* Para salir o terminar el chat en directo, basta con introducir en la celda las palabras `salir`, `fin` o `terminar`.


In [10]:
# Celda Interactiva de Conversacion en Vivo
# Ejecuta este bloque para interactuar y chatear libremente con el Consultor Legal RAG

print("=" * 68)
print("   MODO INTERACTIVO EN VIVO: CONSULTOR LEGAL RAG")
print("   Escribe tu consulta normativa o introduce 'salir' para terminar")
print("=" * 68 + "\n")

historial_interactivo = []

while True:
    try:
        pregunta_usuario = input("\nTú (Escribe tu pregunta al RAG): ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\n[INFO] Sesion interactiva finalizada por orden de interrupcion del usuario.")
        break
        
    if not pregunta_usuario:
        continue
        
    if pregunta_usuario.lower() in ["salir", "exit", "quit", "terminar", "fin"]:
        print("\n[INFO] Cerrando el chat interactivo. Gracias por utilizar el sistema RAG.")
        break
        
    print(f"\n> Consultando base vectorial y procesando grafo para: '{pregunta_usuario}'...")
    
    historial_interactivo.append(HumanMessage(content=pregunta_usuario))
    estado_input = {"messages": historial_interactivo}
    
    t_ini = time.time()
    try:
        estado_salida = app_rag.invoke(estado_input)
        t_tot = time.time() - t_ini
        
        historial_interactivo = estado_salida["messages"]
        respuesta_agente = historial_interactivo[-1].content
        
        print(f"\n--- Respuesta del Consultor Legal RAG ({t_tot:.2f}s) ---")
        texto_md_agente = respuesta_agente if isinstance(respuesta_agente, str) else "".join([b.get("text", "") if isinstance(b, dict) else str(b) for b in respuesta_agente])
        display(Markdown(texto_md_agente))
        print("-" * 68)
    except Exception as exc:
        print(f"\n[ADVERTENCIA] Ha ocurrido un error temporal de red o cuota al consultar la API: {exc}")
        print("Sugerencia: Si la API de Google esta saturada en este instante (Error 429/503), espera unos segundos y vuelve a preguntar.")


   MODO INTERACTIVO EN VIVO: CONSULTOR LEGAL RAG
   Escribe tu consulta normativa o introduce 'salir' para terminar


> Consultando base vectorial y procesando grafo para: '¿En qué supuestos concretos es obligatorio designar un Delegado de Protección de Datos (DPO) según el RGPD y cuáles son sus funciones mínimas exigidas?'...

--- Respuesta del Consultor Legal RAG (2.46s) ---


### Conclusión Directa
La designación de un Delegado de Protección de Datos (DPO) es obligatoria en supuestos específicos vinculados a la naturaleza pública del responsable o a la escala y tipo de tratamiento de datos, siendo sus funciones principales el asesoramiento, la supervisión del cumplimiento y la interlocución con los interesados.

### Análisis Normativo
La obligatoriedad de designar un DPO se circunscribe a los siguientes escenarios:
* **Autoridades u organismos públicos:** Se requiere su designación siempre que el tratamiento sea realizado por estas entidades, con la excepción de los tribunales en el ejercicio de su función judicial.
* **Observación habitual y sistemática:** Cuando las actividades principales del responsable o encargado requieran una observación habitual y sistemática de interesados a gran escala.
* **Tratamiento de categorías especiales de datos:** Cuando las actividades principales impliquen el tratamiento a gran escala de categorías especiales de datos (artículo 9) o datos relativos a condenas e infracciones penales (artículo 10).
* **Previsión normativa adicional:** También será obligatorio cuando así lo exija el Derecho de la Unión o de los Estados miembros.

Las funciones mínimas exigidas al DPO incluyen:
* **Informar y asesorar:** Debe orientar al responsable, al encargado y a los empleados sobre las obligaciones derivadas del RGPD y otras disposiciones de protección de datos.
* **Supervisar el cumplimiento:** Debe velar por el cumplimiento normativo y de las políticas internas, incluyendo la formación del personal, la asignación de responsabilidades y la ejecución de auditorías.
* **Interlocución:** Actuar como punto de contacto para los interesados en cuestiones relativas al tratamiento de sus datos y al ejercicio de sus derechos.

### Trazabilidad Jurídica
* **reglamentoRGPD.md (REGLAMENTOS -> Designación del delegado de protección de datos)** — Supuestos obligatorios de designación.
* **reglamentoRGPD.md (REGLAMENTOS -> Funciones del delegado de protección de datos)** — Funciones mínimas exigidas.
* **reglamentoRGPD.md (REGLAMENTOS -> Posición del delegado de protección de datos)** — Función de interlocución con los interesados.

--------------------------------------------------------------------

> Consultando base vectorial y procesando grafo para: '¿Y qué posición jerárquica o garantías de independencia exige la ley para que esa figura que acabas de explicar pueda desempeñar su trabajo sin recibir instrucciones ni represalias?'...

--- Respuesta del Consultor Legal RAG (3.27s) ---


### Conclusión Directa
El RGPD garantiza la independencia del Delegado de Protección de Datos (DPO) mediante la prohibición de recibir instrucciones sobre el ejercicio de sus funciones, la protección frente a sanciones o destituciones por su desempeño, y la obligación de rendir cuentas directamente al nivel jerárquico más alto de la organización.

### Análisis Normativo
Para asegurar la independencia y eficacia del DPO, el marco normativo establece las siguientes garantías:

* **Autonomía funcional:** El responsable y el encargado del tratamiento tienen la obligación de garantizar que el DPO no reciba ninguna instrucción respecto al desempeño de sus funciones.
* **Protección contra represalias:** El DPO goza de una protección especial, por la cual no puede ser destituido ni sancionado por el responsable o el encargado del tratamiento como consecuencia del ejercicio de sus cometidos.
* **Reporte jerárquico:** Con el fin de asegurar su independencia, el DPO debe rendir cuentas directamente al más alto nivel jerárquico de la organización (ej. dirección general o consejo de administración).
* **Participación y recursos:** El responsable y el encargado deben asegurar que el DPO participe de forma adecuada y oportuna en todas las cuestiones relativas a la protección de datos, facilitándole los recursos necesarios, el acceso a las operaciones de tratamiento y el mantenimiento de sus conocimientos especializados.
* **Confidencialidad:** El DPO está sujeto al deber de secreto o confidencialidad en el desempeño de sus funciones, conforme al Derecho de la Unión o de los Estados miembros.
* **Capacidad operativa:** Independientemente de si es empleado o externo, el DPO debe estar en condiciones de desempeñar sus funciones de manera independiente, ajustando su nivel de conocimientos especializados a la naturaleza de las operaciones de tratamiento.

### Trazabilidad Jurídica
* **reglamentoRGPD.md (REGLAMENTOS -> Posición del delegado de protección de datos)** — Garantías de independencia, prohibición de instrucciones, protección contra represalias, reporte jerárquico, participación, recursos y confidencialidad.
* **reglamentoRGPD.md (REGLAMENTOS -> de 27 de abril de 2016)** — Requisito de independencia y adecuación de conocimientos especializados.

--------------------------------------------------------------------

> Consultando base vectorial y procesando grafo para: '¿En qué consiste exactamente "La Regla de 2" en el diseño de agentes de IA y por qué la normativa advierte que es un error pretender solucionar los problemas de diseño desplazando toda la responsabilidad a la supervisión humana?'...

--- Respuesta del Consultor Legal RAG (3.99s) ---


...

--------------------------------------------------------------------

> Consultando base vectorial y procesando grafo para: '¿Cuáles son los equipos favoritos y las cuotas pronosticadas para ganar la próxima Liga de Campeones de fútbol en Europa?'...

--- Respuesta del Consultor Legal RAG (2.23s) ---


No estoy entrenado para responder sobre ese tema.

--------------------------------------------------------------------

> Consultando base vectorial y procesando grafo para: '¿Qué incentivos o deducciones fiscales en el impuesto sobre la renta ofrece la legislación federal de Canadá para las startups que desarrollan algoritmos de inteligencia artificial?'...

--- Respuesta del Consultor Legal RAG (1.70s) ---


La información disponible en la base de conocimientos no permite responder a esta consulta.

--------------------------------------------------------------------

[INFO] Cerrando el chat interactivo. Gracias por utilizar el sistema RAG.
